<a href="https://colab.research.google.com/github/Johnkfilho/unsupervised-learning/blob/main/10_Embeddings_Analysis_John_Kennedy_20210066166.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Embeddings e Redução da Dimensionalidade

**Objetivo.** Dado um conjunto de textos, gerar embeddings com BERT e investigar a estrutura dos dados via PCA, t-SNE e UMAP. Em seguida, identificar clusters e relacioná-los a categorias semânticas.

In [36]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap.umap_ as umap
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns

In [37]:
sentences = [
    'I swap butter for olive oil in many recipes.',
    'Canberra is the capital of Australia.',
    'Ottawa is the capital city of Canada.',
    'Paris is the most populated city in France.',
    'Tokyo is among the most populous metropolitan areas worldwide.',
    'I prefer my coffee with no sugar and a splash of milk.',
    'The recipe for pasta carbonara is simple.',
    'A pinch of salt enhances sweetness in desserts.',
    'Alignment techniques reduce harmful outputs.',
    'Explainable AI highlights salient features for decisions.',
    'Transformer models enable long-range language dependencies.',
    'Black swan events stress-test portfolio resilience.',
    'The Sahara Desert spans much of North Africa.',
    'Inflation erodes real purchasing power of cash.',
    'Aromatics like garlic and onion build flavor early.',
    'Value stocks trade at lower multiples relative to fundamentals.',
    'Quantization reduces memory with minimal accuracy loss.',
    'Tax-loss harvesting offsets capital gains.',
    'Investing in technology can be risky.',
    'Fermented foods add acidity and complexity.',
    'Marinating tofu improves texture and taste.',
    'Vector databases power semantic search at scale.',
    'Distillation transfers knowledge from large to small models.',
    'The Great Barrier Reef lies off Australia’s northeast coast.',
    'Retrieval-augmented generation grounds answers in sources.',
    'Iceland lies on the Mid-Atlantic Ridge.',
    'The Baltic states border the eastern Baltic Sea.',
    'Multimodal learning aligns text with images and audio.',
    'Risk tolerance should guide position sizing.',
    'Time in the market beats timing the market.',
    'Behavioral biases can derail investment plans.',
    'Reinforcement learning fine-tunes policies from human feedback.',
    'Edge AI runs models under strict latency constraints.',
    'Deglazing lifts browned bits to make pan sauces.',
    'Tempering chocolate stabilizes cocoa butter crystals.',
    'What is the capital of France?',
    'Johannesburg is a major city but not South Africa’s capital.',
    'The Danube passes through multiple European capitals.',
    'The Amazon River carries one of the largest water volumes on Earth.',
    'A healthy emergency fund reduces forced selling.',
    'I batch-cook grains for quick lunches.',
    'Resting steak helps redistribute the juices.',
    'The Atacama is one of the driest deserts on the planet.',
    'Liquidity risk rises when trading volumes are thin.',
    'Mount Everest is the highest peak above sea level.',
    'Graph neural networks capture relational structure.',
    'Sourdough starter needs regular feedings to stay active.',
    'The stock market experienced a drop today.',
    'Umami-rich ingredients deepen savory dishes.',
    'Al dente pasta retains a slight bite after cooking.',
    'Rebalancing restores target asset allocation.',
    'Continual learning mitigates catastrophic forgetting.',
    'Bond duration measures sensitivity to interest-rate changes.',
    'Diffusion models synthesize high-fidelity images.',
    'Expense ratios compound against long-term returns.',
    'Self-supervised pretraining reduces labeled data needs.',
    'What country contains the city of Kyoto?',
    'Stir-frying requires high heat and constant movement.',
    'Covered calls generate income with capped upside.',
    'The Nile flows northward into the Mediterranean Sea.',
    'Causal inference distinguishes correlation from effect.',
    'Prompt engineering steers generative behavior reliably.',
    'Few-shot prompting improves generalization on new tasks.',
    'Growth investing prioritizes earnings expansion.',
    'The Alps stretch across several central European countries.',
    'The Andes form a continuous mountain range along South America.',
    'I cook vegetarian meals on weekdays to simplify planning.',
    'Natural language processing has advanced greatly.',
    'Sous-vide delivers precise temperature control.',
    'Diversification reduces idiosyncratic risk across holdings.',
    'Sharpe ratio evaluates risk-adjusted performance.',
    'Artificial intelligence is transforming the world.',
    'Credit spreads widen during economic uncertainty.',
    'Emerging markets add diversification but higher volatility.',
    'Mise en place speeds up weeknight cooking.',
    'The Caspian Sea is a landlocked body of water.',
    'Evaluation with benchmarks must avoid data leakage.',
    'Cairo sits along the Nile River delta.',
    'Federated learning trains models without centralizing data.',
    'Lagos is Nigeria’s largest city by population.',
    'Dollar-cost averaging smooths entry price over time.',
    'LoRA adapters enable efficient fine-tuning.',
    'I keep a jar of homemade pesto for pasta.',
    'New Delhi serves as the seat of India’s government.',
    'I like to cook Italian dishes on Sundays.',
    'Roasting vegetables caramelizes natural sugars.',
    'ETFs provide broad market exposure with intraday liquidity.',
    'Proofing time affects a bread’s crumb structure.'
]

## Predição dos Embeddings

Utilize o modelo BERT pré-treinado para gerar embeddings de todos os textos fornecidos.  
O objetivo é obter uma matriz `X` com formato **(N, dim)**, onde **N** é o número de textos e **dim** é a dimensionalidade dos vetores de embedding.

In [38]:
model = SentenceTransformer('all-MiniLM-L6-v2')
X = model.encode(sentences)

N = X.shape[0]
dim = X.shape[1]

print("\n--- Resultado ---")
print(f"Número de textos (N): {N}")
print(f"Dimensionalidade dos embeddings (dim): {dim}")
print(f"Formato da matriz X: ({N}, {dim})")
print(f"Conteúdo de X (Primeiras 5 linhas, primeiras 5 dimensões):\n{X[:5, :5]}")

np.save('bert_embeddings_X.npy', X)
print("\nMatriz de embeddings salva como 'bert_embeddings_X.npy'.")


--- Resultado ---
Número de textos (N): 88
Dimensionalidade dos embeddings (dim): 384
Formato da matriz X: (88, 384)
Conteúdo de X (Primeiras 5 linhas, primeiras 5 dimensões):
[[-0.05245963 -0.07178463  0.0277141   0.01968163  0.00380771]
 [ 0.07836344  0.01278537 -0.02111468  0.0718139  -0.02776916]
 [ 0.05620829  0.0233661   0.00178018  0.01617255  0.01801894]
 [ 0.11008075 -0.02245203  0.00810891 -0.0460946   0.07291324]
 [ 0.14343841 -0.06150825  0.06349361  0.02244709 -0.00363488]]

Matriz de embeddings salva como 'bert_embeddings_X.npy'.


## PCA

Aplique **PCA (Principal Component Analysis)** para projetar os embeddings em duas dimensões e visualizar a estrutura global dos dados.  
O PCA ajuda a capturar as direções de maior variância e pode indicar agrupamentos lineares.

**Tarefas:**
- Reduza a dimensionalidade dos embeddings para 2 componentes principais.  
- Plote os pontos resultantes com `matplotlib`, identificando possíveis agrupamentos.  
- Analise qualitativamente se há separação entre textos de temas distintos.

In [39]:
try:
    X = np.load('bert_embeddings_X.npy')
    print(f"Embeddings (X) carregados com sucesso. Formato: {X.shape}")
except FileNotFoundError:
    print("AVISO: Arquivo 'bert_embeddings_X.npy' não encontrado. Gerando dados de simulação (PCA demonstrativo).")
    N = 88
    dim = 384
    X = np.random.rand(N, dim)

pca = PCA(n_components=2, random_state=42)
X_pca_2d = pca.fit_transform(X)

df_pca = pd.DataFrame(data=X_pca_2d, columns=['PC1', 'PC2'])
df_pca['sentence'] = sentences

variance_ratio = pca.explained_variance_ratio_.sum()

print(f"\nVariância total explicada pelos 2 componentes principais: {variance_ratio:.2%}")
print(df_pca.head())

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PC1',
    y='PC2',
    data=df_pca,
    s=70,
    alpha=0.7,
    palette='viridis'
)

plt.title(f'PCA de Embeddings BERT (2 Componentes)\nVariância Explicada: {variance_ratio:.2%}')
plt.xlabel('Componente Principal 1 (PC1)')
plt.ylabel('Componente Principal 2 (PC2)')
plt.grid(True, linestyle='--', alpha=0.5)

extreme_points_indices = [
    df_pca['PC1'].idxmax(),
    df_pca['PC1'].idxmin(),
    df_pca['PC2'].idxmax(),
    df_pca['PC2'].idxmin()
]

for i in np.unique(extreme_points_indices):
    plt.annotate(
        df_pca.iloc[i]['sentence'][:20] + '...', # Primeira parte da frase
        (df_pca.iloc[i]['PC1'], df_pca.iloc[i]['PC2']),
        textcoords="offset points", xytext=(5,5), ha='center', fontsize=8, color='red'
    )

plt.savefig('pca_bert_embeddings_2d.png')
plt.close()

print("\nGráfico PCA 2D salvo como 'pca_bert_embeddings_2d.png'.")

Embeddings (X) carregados com sucesso. Formato: (88, 384)

Variância total explicada pelos 2 componentes principais: 14.12%
        PC1       PC2                                           sentence
0 -0.240903 -0.286988       I swap butter for olive oil in many recipes.
1 -0.139788  0.350374              Canberra is the capital of Australia.
2 -0.158964  0.337041              Ottawa is the capital city of Canada.
3 -0.246618  0.368071        Paris is the most populated city in France.
4 -0.086549  0.411557  Tokyo is among the most populous metropolitan ...


/tmp/ipython-input-3400113803.py:22: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  sns.scatterplot(



Gráfico PCA 2D salvo como 'pca_bert_embeddings_2d.png'.


## t-SNE

Use **t-SNE (t-distributed Stochastic Neighbor Embedding)** para investigar a estrutura local dos dados.  
Diferente do PCA, o t-SNE tenta preservar vizinhanças locais e pode revelar grupos mais sutis.

**Tarefas:**
- Reduza os embeddings para 2D usando `TSNE` do `scikit-learn`.  
- Ajuste parâmetros como `perplexity` e `learning_rate` para comparar resultados.  
- Visualize o mapa e observe se os textos semelhantes ficam próximos.

In [40]:
try:
    X = np.load('bert_embeddings_X.npy')
    print(f"Embeddings (X) carregados com sucesso. Formato: {X.shape}")
except FileNotFoundError:
    print("AVISO: Arquivo 'bert_embeddings_X.npy' não encontrado. Gerando dados de simulação.")
    N = 88
    dim = 384
    X = np.random.rand(N, dim)

n_pca_components = min(50, X.shape[0] - 1)
pca = PCA(n_components=n_pca_components, random_state=42)
X_pca = pca.fit_transform(X)
print(f"Redução prévia com PCA para {n_pca_components} dimensões concluída.")

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    random_state=42,
    n_jobs=-1
)
X_tsne_2d = tsne.fit_transform(X_pca)

df_tsne = pd.DataFrame(data=X_tsne_2d, columns=['TSNE_1', 'TSNE_2'])
df_tsne['sentence'] = sentences

print(df_tsne.head())

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='TSNE_1',
    y='TSNE_2',
    data=df_tsne,
    s=70,
    alpha=0.7,
    palette='Spectral'
)

plt.title(f't-SNE de Embeddings BERT (Perplexity=30)')
plt.xlabel('t-SNE Componente 1')
plt.ylabel('t-SNE Componente 2')
plt.grid(True, linestyle='--', alpha=0.5)

indices_rep = [6, 15, 1, 9]

for i in indices_rep:
    plt.annotate(
        df_tsne.iloc[i]['sentence'][:20] + '...',
        (df_tsne.iloc[i]['TSNE_1'], df_tsne.iloc[i]['TSNE_2']),
        textcoords="offset points", xytext=(5,5), ha='center', fontsize=8, color='red'
    )

plt.savefig('tsne_bert_embeddings_2d.png')
plt.close()

print("\nGráfico t-SNE 2D salvo como 'tsne_bert_embeddings_2d.png'.")

Embeddings (X) carregados com sucesso. Formato: (88, 384)
Redução prévia com PCA para 50 dimensões concluída.
     TSNE_1    TSNE_2                                           sentence
0 -2.853675  1.484306       I swap butter for olive oil in many recipes.
1  5.337320  2.853542              Canberra is the capital of Australia.
2  4.665442  2.782843              Ottawa is the capital city of Canada.
3  3.011679  3.245188        Paris is the most populated city in France.
4  4.715577  4.449505  Tokyo is among the most populous metropolitan ...


/tmp/ipython-input-1312290716.py:31: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  sns.scatterplot(



Gráfico t-SNE 2D salvo como 'tsne_bert_embeddings_2d.png'.


## UMAP

Aplique **UMAP (Uniform Manifold Approximation and Projection)** como alternativa ao t-SNE.  
O UMAP é mais eficiente, preserva parte da estrutura global e é útil para visualização e pré-processamento.

**Tarefas:**
- Gere uma projeção 2D dos embeddings com `umap.UMAP`.  
- Experimente variar `n_neighbors` e `min_dist` para observar mudanças na distribuição dos clusters.  
- Compare visualmente com os resultados do PCA e t-SNE.

In [41]:
try:
    X = np.load('bert_embeddings_X.npy')
    print(f"Embeddings (X) carregados com sucesso. Formato: {X.shape}")
except FileNotFoundError:
    print("AVISO: Arquivo 'bert_embeddings_X.npy' não encontrado. Gerando dados de simulação.")
    N = 88
    dim = 384
    X = np.random.rand(N, dim)

n_pca_components = min(50, X.shape[0] - 1)
pca = PCA(n_components=n_pca_components, random_state=42)
X_pca = pca.fit_transform(X)
print(f"Redução prévia com PCA para {n_pca_components} dimensões concluída.")

reducer_umap = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=42
)

X_umap_2d = reducer_umap.fit_transform(X_pca)

df_umap = pd.DataFrame(data=X_umap_2d, columns=['UMAP_1', 'UMAP_2'])
df_umap['sentence'] = sentences

print(df_umap.head())

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='UMAP_1',
    y='UMAP_2',
    data=df_umap,
    s=70,
    alpha=0.7,
    palette='viridis'
)

plt.title(f'UMAP de Embeddings BERT (Vizinhos={reducer_umap.n_neighbors}, Distância Mínima={reducer_umap.min_dist})')
plt.xlabel('UMAP Componente 1')
plt.ylabel('UMAP Componente 2')
plt.grid(True, linestyle='--', alpha=0.5)

indices_rep = [6, 15, 1, 9]

for i in indices_rep:
    plt.annotate(
        df_umap.iloc[i]['sentence'][:20] + '...',
        (df_umap.iloc[i]['UMAP_1'], df_umap.iloc[i]['UMAP_2']),
        textcoords="offset points", xytext=(5,5), ha='center', fontsize=8, color='red'
    )

plt.savefig('umap_bert_embeddings_2d.png')
plt.close()

print("\nGráfico UMAP 2D salvo como 'umap_bert_embeddings_2d.png'.")

Embeddings (X) carregados com sucesso. Formato: (88, 384)
Redução prévia com PCA para 50 dimensões concluída.


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


     UMAP_1    UMAP_2                                           sentence
0  3.638671 -1.711680       I swap butter for olive oil in many recipes.
1  0.488907 -2.120105              Canberra is the capital of Australia.
2  0.897214 -1.912992              Ottawa is the capital city of Canada.
3  0.948639 -2.385879        Paris is the most populated city in France.
4  1.100971 -2.443476  Tokyo is among the most populous metropolitan ...


/tmp/ipython-input-2592226703.py:33: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.
  sns.scatterplot(



Gráfico UMAP 2D salvo como 'umap_bert_embeddings_2d.png'.


## Classificação

Com base nas categorias observadas nos gráficos anteriores, crie uma função simples que receba um texto e classifique-o na categoria mais provável.

**Tarefas:**
- Use os embeddings existentes e os clusters identificados para rotular automaticamente cada texto.  
- Crie uma função `classificar_texto(texto: str)` que:
  1. Gere o embedding do texto.
  2. Calcule a distância para os clusters identificados.
  3. Retorne o nome do cluster mais próximo.

In [42]:
try:
    X_original = np.load('bert_embeddings_X.npy')
    print(f"Embeddings (X) carregados. Formato: {X_original.shape}")
except FileNotFoundError:
    print("AVISO: Usando dados simulados para demonstração da função de classificação.")
    N = 88
    dim = 384
    X_original = np.random.rand(N, dim)

sentences = [
    'I swap butter for olive oil in many recipes.', 'Canberra is the capital of Australia.',
    'Alignment techniques reduce harmful outputs.', 'Black swan events stress-test portfolio resilience.',
]

n_pca_components = min(30, X_original.shape[0] - 1)
pca = PCA(n_components=n_pca_components, random_state=42)
X_pca = pca.fit_transform(X_original)

N_CLUSTERS = 4
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X_pca)
centroids_pca = kmeans.cluster_centers_

cluster_names = {
    0: 'CULINÁRIA/RECEITAS',
    1: 'GEOGRAFIA/MUNDO',
    2: 'FINANÇAS/INVESTIMENTOS',
    3: 'IA/ML/TECNOLOGIA'
}

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modelo BERT inicializado para classificação.")

def classificar_texto(texto: str) -> str:
    """
    Classifica um novo texto em uma das categorias semânticas (clusters)
    encontradas na análise de embeddings.

    Args:
        texto (str): O novo texto a ser classificado.

    Returns:
        str: O nome da categoria semântica mais provável.
    """

    new_embedding = model.encode([texto])

    new_embedding_pca = pca.transform(new_embedding)

    distances = cdist(new_embedding_pca, centroids_pca, metric='euclidean')

    closest_cluster_index = np.argmin(distances)

    return cluster_names.get(closest_cluster_index, 'CLUSTER DESCONHECIDO')

print("\n--- TESTANDO A FUNÇÃO CLASSIFICAR_TEXTO ---")

teste_1 = "Optimizing models via reinforcement learning from human feedback is crucial."
print(f"Texto: \"{teste_1}\" -> Categoria: {classificar_texto(teste_1)}")

teste_2 = "Searing the scallops creates a nice caramelized crust."
print(f"Texto: \"{teste_2}\" -> Categoria: {classificar_texto(teste_2)}")

teste_3 = "Diversifying assets reduces non-systematic risk in a portfolio."
print(f"Texto: \"{teste_3}\" -> Categoria: {classificar_texto(teste_3)}")

teste_4 = "The largest volcano in Europe is Mount Etna in Sicily."
print(f"Texto: \"{teste_4}\" -> Categoria: {classificar_texto(teste_4)}")

Embeddings (X) carregados. Formato: (88, 384)
Modelo BERT inicializado para classificação.

--- TESTANDO A FUNÇÃO CLASSIFICAR_TEXTO ---
Texto: "Optimizing models via reinforcement learning from human feedback is crucial." -> Categoria: IA/ML/TECNOLOGIA
Texto: "Searing the scallops creates a nice caramelized crust." -> Categoria: CULINÁRIA/RECEITAS
Texto: "Diversifying assets reduces non-systematic risk in a portfolio." -> Categoria: IA/ML/TECNOLOGIA
Texto: "The largest volcano in Europe is Mount Etna in Sicily." -> Categoria: GEOGRAFIA/MUNDO
